# 01 · Explore the data

Loads SPY from the local store, compares adjusted and unadjusted closes, and shows the
data-quality summary. Run `uv run qr init && uv run qr ingest` first.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from qr import storage, validate
from qr.config import db_path
from qr.load import get_bars, get_returns, list_symbols

# The notebook lives in notebooks/, the data directory one level up.
DATA_DIR = Path(os.environ.get("QR_DATA_DIR", Path.cwd().parent / "data"))
list_symbols(data_dir=DATA_DIR)

## SPY in the three adjustment modes

`get_bars` returns a `(date, symbol)` MultiIndex. Prices are stored unadjusted; split and
total-return adjustment are computed at load time.

In [ ]:
spy = {
    mode: get_bars("SPY", adjustment=mode, data_dir=DATA_DIR).droplevel("symbol")
    for mode in ("none", "split", "total_return")
}
spy["total_return"].tail()

In [ ]:
closes = pd.DataFrame({mode: frame["close"] for mode, frame in spy.items()})
closes.iloc[[0, len(closes) // 2, -1]]

## Adjusted vs. unadjusted close

SPY has never split, so the whole difference is dividends: the total-return series starts
lower because every dividend paid since then has been folded back into earlier prices.

In [ ]:
fig, (top, bottom) = plt.subplots(2, 1, figsize=(10, 7), sharex=True, height_ratios=[3, 1])

top.plot(closes.index, closes["none"], label="Unadjusted close", linewidth=1.2)
top.plot(closes.index, closes["total_return"], label="Total-return adjusted close", linewidth=1.2)
top.set_yscale("log")
top.set_ylabel("USD (log scale)")
top.set_title("SPY: unadjusted vs. total-return adjusted close")
top.legend(frameon=False)
top.grid(alpha=0.3)

bottom.plot(closes.index, closes["total_return"] / closes["none"], color="tab:gray", linewidth=1.2)
bottom.set_ylabel("Adjusted / unadjusted")
bottom.set_title("Cumulative dividend adjustment factor", fontsize=10)
bottom.grid(alpha=0.3)

fig.tight_layout()
plt.show()

What dividends are worth over the whole period: price-only growth versus total return.

In [ ]:
growth = closes.iloc[-1] / closes.iloc[0]
pd.DataFrame(
    {"growth multiple": growth.round(2)},
    index=["none", "total_return"],
).rename(index={"none": "price only", "total_return": "with dividends reinvested"})

## Validation summary

Open findings in `quality_issues`, grouped by symbol and check, plus per-symbol coverage.
An empty table means every check passed.

In [ ]:
with storage.connect(db_path(DATA_DIR), read_only=True) as con:
    issues = storage.read_quality_issues(con)
    coverage = storage.read_coverage(con)

print(f"{len(issues)} open findings")
validate.summarize(issues)

In [ ]:
coverage

## Daily returns

`get_returns` gives a wide (date × symbol) table of total returns (log by default).

In [ ]:
returns = get_returns(["SPY", "TLT", "GLD"], kind="simple", data_dir=DATA_DIR)
returns.describe().loc[["count", "mean", "std", "min", "max"]]